# CUDA Kernel 面试主线 · 第 4/12 课：RMSNorm：统计量与两遍访问

> 状态：**参考答案版**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现 RMSNorm 的均方根统计，解释它与 LayerNorm 的差异。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：C/C++、线性代数、基本并行编程
- 本课在路线中的作用：RMSNorm 只按均方根缩放，不减均值；LLM 中常按最后一维做归一化。

## 核心心智模型

### 1. 它是什么，解决什么问题

RMSNorm 只按均方根缩放，不减均值；LLM 中常按最后一维做归一化。

### 2. 它如何工作

第一遍归约 sum(x²)，得到 inv_rms=rsqrt(sum/N+eps)；同步后第二遍乘输入和 weight。

### 3. 正确性条件与常见误区

eps 加在均值之后、开方之前；统计与累积应使用足够精度，且 N 必须大于 0。

### 4. 性能与工程取舍

少一个 sum 统计且无 beta，但仍需读两遍输入；融合残差/量化可减少 HBM 流量。

## 具体演示

输入 [3,4]、eps=0 时 rms=sqrt(12.5)，输出按同一个 inv_rms 缩放。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐 inverse RMS。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/04_rmsnorm.cu
#include <cuda_runtime.h>

namespace {

// warp reduce: 用 shuffle 在寄存器之间传值，避免 shared memory 往返。
template<int BLOCK_SIZE>
__device__ __forceinline__ float warp_reduce_sum(float val) {
    #pragma unroll
    for (int offset = 16; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xffffffff, val, offset);
    }
    return val;
}

// block reduce: warp 内 reduce + shared memory 跨 warp 汇总。
template<int BLOCK_SIZE>
__device__ __forceinline__ float block_reduce_sum(float val) {
    constexpr int NUM_WARPS = (BLOCK_SIZE + 31) / 32;
    __shared__ float smem[NUM_WARPS];

    int lane = threadIdx.x & 31;
    int warp = threadIdx.x >> 5;

    val = warp_reduce_sum<BLOCK_SIZE>(val);

    if (lane == 0) {
        smem[warp] = val;
    }
    __syncthreads();

    val = (threadIdx.x < NUM_WARPS) ? smem[lane] : 0.0f;
    if (warp == 0) {
        val = warp_reduce_sum<BLOCK_SIZE>(val);
    }

    __syncthreads();
    return val;
}

// RMSNorm: 和 LayerNorm 很像，但不减 mean，也没有 beta。
// input/output: [M, N] row-major
// weight: [N]
//
// 数学公式：
// rms = sqrt(sum(x_i^2) / N + eps)
// y_i = x_i / rms * weight_i
//
// 面试里常问它为什么比 LayerNorm 少一些计算：
// RMSNorm 不需要 sum(x)，只需要 sum(x^2)，所以少一次 reduce，
// 同时输出阶段也少了减 mean 和 beta。
template<int BLOCK_SIZE>
__global__ void rmsnorm_kernel(
    const float* __restrict__ input,
    const float* __restrict__ weight,
    float* __restrict__ output,
    int M,
    int N,
    float eps
) {
    int row = blockIdx.x;
    int tid = threadIdx.x;
    if (row >= M) return;

    const float* row_in = input + row * N;
    float* row_out = output + row * N;

    float local_sq_sum = 0.0f;

    // 每个线程处理若干列，局部累加平方和。
    // 同一轮 col=tid 时，相邻线程访问相邻地址，访存合并。
    for (int col = tid; col < N; col += BLOCK_SIZE) {
        float x = row_in[col];
        local_sq_sum += x * x;
    }

    float sq_sum = block_reduce_sum<BLOCK_SIZE>(local_sq_sum);

    // inv_rms 是这一行所有元素共享的标量。
    __shared__ float smem_inv_rms;

    if (tid == 0) {
        smem_inv_rms = ______;  // TODO: 稳定的 inverse RMS
    }
    __syncthreads();

    float inv_rms = smem_inv_rms;

    // 第二 pass 写输出。这里读 input、读 weight、写 output 都是连续访问。
    for (int col = tid; col < N; col += BLOCK_SIZE) {
        row_out[col] = row_in[col] * inv_rms * weight[col];
    }
}

} // namespace

void launch_rmsnorm(
    const float* input,
    const float* weight,
    float* output,
    int M,
    int N,
    float eps,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    rmsnorm_kernel<BLOCK_SIZE><<<M, BLOCK_SIZE, 0, stream>>>(
        input, weight, output, M, N, eps
    );
}


### 检查方法

有 CUDA 环境时执行 `nvcc -std=c++17 -c /tmp/04_rmsnorm.cu -o /tmp/04_rmsnorm.cu.o`；无 CUDA 环境时只做静态审查并登记待验证。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“RMSNorm：统计量与两遍访问”的工作机制。

**你的答案：**


### Q2

误把 `sum(x*x)` 除以 `sqrt(N)` 会造成什么尺度错误？

**你的答案：**


### Q3

RMSNorm 与 residual add 融合时能省哪些读写？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考答案（仅 answer 分支）

先完成题目再核对。即使代码一致，也要能解释关键步骤，并尝试更换一个输入规模。

In [ ]:
%%writefile /tmp/04_rmsnorm.cu
#include <cuda_runtime.h>

namespace {

// warp reduce: 用 shuffle 在寄存器之间传值，避免 shared memory 往返。
template<int BLOCK_SIZE>
__device__ __forceinline__ float warp_reduce_sum(float val) {
    #pragma unroll
    for (int offset = 16; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xffffffff, val, offset);
    }
    return val;
}

// block reduce: warp 内 reduce + shared memory 跨 warp 汇总。
template<int BLOCK_SIZE>
__device__ __forceinline__ float block_reduce_sum(float val) {
    constexpr int NUM_WARPS = (BLOCK_SIZE + 31) / 32;
    __shared__ float smem[NUM_WARPS];

    int lane = threadIdx.x & 31;
    int warp = threadIdx.x >> 5;

    val = warp_reduce_sum<BLOCK_SIZE>(val);

    if (lane == 0) {
        smem[warp] = val;
    }
    __syncthreads();

    val = (threadIdx.x < NUM_WARPS) ? smem[lane] : 0.0f;
    if (warp == 0) {
        val = warp_reduce_sum<BLOCK_SIZE>(val);
    }

    __syncthreads();
    return val;
}

// RMSNorm: 和 LayerNorm 很像，但不减 mean，也没有 beta。
// input/output: [M, N] row-major
// weight: [N]
//
// 数学公式：
// rms = sqrt(sum(x_i^2) / N + eps)
// y_i = x_i / rms * weight_i
//
// 面试里常问它为什么比 LayerNorm 少一些计算：
// RMSNorm 不需要 sum(x)，只需要 sum(x^2)，所以少一次 reduce，
// 同时输出阶段也少了减 mean 和 beta。
template<int BLOCK_SIZE>
__global__ void rmsnorm_kernel(
    const float* __restrict__ input,
    const float* __restrict__ weight,
    float* __restrict__ output,
    int M,
    int N,
    float eps
) {
    int row = blockIdx.x;
    int tid = threadIdx.x;
    if (row >= M) return;

    const float* row_in = input + row * N;
    float* row_out = output + row * N;

    float local_sq_sum = 0.0f;

    // 每个线程处理若干列，局部累加平方和。
    // 同一轮 col=tid 时，相邻线程访问相邻地址，访存合并。
    for (int col = tid; col < N; col += BLOCK_SIZE) {
        float x = row_in[col];
        local_sq_sum += x * x;
    }

    float sq_sum = block_reduce_sum<BLOCK_SIZE>(local_sq_sum);

    // inv_rms 是这一行所有元素共享的标量。
    __shared__ float smem_inv_rms;

    if (tid == 0) {
        smem_inv_rms = rsqrtf(sq_sum / static_cast<float>(N) + eps);
    }
    __syncthreads();

    float inv_rms = smem_inv_rms;

    // 第二 pass 写输出。这里读 input、读 weight、写 output 都是连续访问。
    for (int col = tid; col < N; col += BLOCK_SIZE) {
        row_out[col] = row_in[col] * inv_rms * weight[col];
    }
}

} // namespace

void launch_rmsnorm(
    const float* input,
    const float* weight,
    float* output,
    int M,
    int N,
    float eps,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    rmsnorm_kernel<BLOCK_SIZE><<<M, BLOCK_SIZE, 0, stream>>>(
        input, weight, output, M, N, eps
    );
}


### Q1 参考答案

第一遍归约 sum(x²)，得到 inv_rms=rsqrt(sum/N+eps)；同步后第二遍乘输入和 weight。

### Q2 参考答案

判断时先检查本课不变量：eps 加在均值之后、开方之前；统计与累积应使用足够精度，且 N 必须大于 0。  若不成立，最终数值或系统状态即使暂时正常也不可信。

### Q3 参考答案

迁移时先保证正确性，再比较代价。这里的核心取舍是：少一个 sum 统计且无 beta，但仍需读两遍输入；融合残差/量化可减少 HBM 流量。

## 参考资料

- [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。